# Regression

 📝 Core Concepts: Regression vs. Classification

When training neural networks, supervised machine learning problems are generally split into two main categories based on what the model is trying to predict:

*   **Regression (This Notebook):** Predicting a continuous **numerical value or number**. 
    *   *Examples:* Predicting house prices, temperature variations, stock values, or mapping a mathematical line (y = x + 10).
    *   *Core Question:* *"How much or how many?"*
    *   *Typical Output Layer:* A single neuron (`nn.Linear(in_features, out_features=1)`) without an activation function, outputting a raw scalar number.

*   **Classification:** Predicting a discrete **label, category, or class**.
    *   *Examples:* Identifying if an email is spam or not spam (binary), or sorting an image into a cat, dog, or bird (multi-class).
    *   *Core Question:* *"Which category does this belong to?"*
    *   *Typical Output Layer:* One or more neurons paired with a classification activation function (like Sigmoid or Softmax) to output probabilities.


In [ ]:
# import
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
from torchinfo import summary # Modern replacement for tf.keras model.summary()

print(torch.__version__)


In [ ]:
# create features (converted to PyTorch Tensors)
x = torch.tensor([-7.0, -4.0, -1.0, 2.0, 5.0, 8.0, 11.0, 14.0])

# create labels
y = torch.tensor([3.0, 6.0, 9.0, 12.0, 15.0, 18.0, 21.0, 24.0])

x.shape, y.shape

# visualize
plt.scatter(x.numpy(), y.numpy())
plt.show()

## model traning

In [ ]:
# 0. Set random seed for reproducibility
torch.manual_seed(42)

# Create features and labels (Ensure float32 and shape [8, 1] for PyTorch)
x = torch.tensor([-7.0, -4.0, -1.0, 2.0, 5.0, 8.0, 11.0, 14.0], dtype=torch.float32).unsqueeze(dim=1)
y = torch.tensor([3.0, 6.0, 9.0, 12.0, 15.0, 18.0, 21.0, 24.0], dtype=torch.float32).unsqueeze(dim=1)

# 1. Create model using nn.Sequential
model = nn.Sequential(
    nn.Linear(in_features=1, out_features=1)
)

# 2. Define loss function (MAE) and Optimizer (SGD)
loss_fn = nn.L1Loss() # L1Loss is Mean Absolute Error (MAE)
optimizer = optim.SGD(model.parameters(), lr=0.001) # Added a small learning rate

# 3. Fit the model (PyTorch requires a manual training loop)
epochs = 5
for epoch in range(epochs):
    model.train()
    
    # 1. Forward pass
    y_pred = model(x)
    
    # 2. Calculate loss
    loss = loss_fn(y_pred, y)
    
    # 3. Optimizer zero grad
    optimizer.zero_grad()
    
    # 4. Backward pass (backpropagation)
    loss.backward()
    
    # 5. Update weights
    optimizer.step()
    
    print(f"Epoch: {epoch+1} | Loss: {loss.item():.4f}")

# 4. Let's see model prediction again for 17.0
model.eval() # Put model in evaluation mode
with torch.inference_mode(): # Turns off gradient tracking for faster predictions
    test_x = torch.tensor([[17.0]], dtype=torch.float32)
    prediction = model(test_x)
    print(f"\nPrediction for 17.0: {prediction.item():.4f}")

# improving the model

In [ ]:
# 2. Re-initialize Optimizer with a better Learning Rate
optimizer = optim.SGD(model.parameters(), lr=0.01) 

# 3. Fit the model for more epochs
epochs = 100 # Increased from 5 to 100
for epoch in range(epochs):
    model.train()
    
    y_pred = model(x)
    loss = loss_fn(y_pred, y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch: {epoch+1} | Loss: {loss.item():.4f}")

# 4. Predict again
model.eval()
with torch.inference_mode():
    test_x = torch.tensor([[17.0]], dtype=torch.float32)
    prediction = model(test_x)
    print(f"\nNew Prediction for 17.0: {prediction.item():.4f}")


## bigger model

In [ ]:
# Make a bigger dataset (using torch.arange)
x = torch.arange(-100, 100, 4, dtype=torch.float32)
x

In [ ]:
# Make label for data set
y = x + 10
y

In [ ]:
# visualise data
plt.scatter(x,y)

In [ ]:
len(x)

In [ ]:
# Reshape into a 2D matrix (Nx1) for PyTorch neural network compatability
x = x.unsqueeze(dim=1)
y = y.unsqueeze(dim=1)

# Split data to train and test sets (80% / 20%)
x_train = x[:40] 
y_train = y[:40]

x_test = x[40:]
y_test = y[40:]

print(f"Train size: {len(x_train)} | Test size: {len(x_test)}")

In [ ]:
# Visualize the training and test data
plt.figure(figsize=(10, 7))
# Convert tensors back to numpy arrays for plotting
plt.scatter(x_train.numpy(), y_train.numpy(), c="b", label="Training data")
plt.scatter(x_test.numpy(), y_test.numpy(), c="g", label="Testing data")
plt.legend()
plt.show()

### train big model

In [ ]:
# --- SETUP REPRODUCIBILITY & DATA ---
torch.manual_seed(42)

# Assuming x_train, y_train, x_test, y_test from earlier steps
# Example shapes: (40, 1) and (10, 1)
# ------------------------------------

# ==========================================
# MODEL 1: Single Node
# ==========================================
model_1 = nn.Sequential(
    nn.Linear(in_features=1, out_features=1)
)
print("--- Model 1 Summary ---")
summary(model_1, input_size=(40, 1))

In [ ]:
# ==========================================
# MODEL 2: Single Layer with 10 Hidden Units
# ==========================================
model_2 = nn.Sequential(
    nn.Linear(in_features=1, out_features=10)
)
print("\n--- Model 2 Summary ---")
summary(model_2, input_size=(40, 1))

In [ ]:
# ==========================================
# MODEL 3: Two layers with custom names (nn.Module style)
# ==========================================
# PyTorch OrderedDict or class styles handle named blocks 
# to replicate TF's `name=` layer arguments.
from collections import OrderedDict

my_model = nn.Sequential(OrderedDict([
    ('input_layer_myin_1', nn.Linear(in_features=1, out_features=50)),
    ('output_layer_myout_1', nn.Linear(in_features=50, out_features=1))
]))

print("\n--- Model 3 (My Model) Summary ---")
summary(my_model, input_size=(40, 1))

In [ ]:
# ==========================================
# COMPILE & FIT (Training My Model)
# ==========================================
loss_fn = nn.L1Loss() # MAE Loss
optimizer = optim.SGD(my_model.parameters(), lr=0.01)

epochs = 100
for epoch in range(epochs):
    my_model.train()
    
    # 1. Forward Pass
    y_pred_train = my_model(x_train)
    
    # 2. Calculate Loss
    loss = loss_fn(y_pred_train, y_train)
    
    # 3. Optimize Steps
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Verbose = 1 equivalent logging
    print(f"Epoch {epoch+1}/{epochs} - loss: {loss.item():.4f} - mae: {loss.item():.4f}")


In [ ]:
# ==========================================
# EVALUATE / PREDICT
# ==========================================
my_model.eval()
with torch.inference_mode():
    # Make predictions on test data
    y_pred = my_model(x_test)


# ==========================================
# PLOTTING FUNCTION
# ==========================================
def plot_predictions(train_data=x_train,
                     train_labels=y_train,
                     test_data=x_test,
                     test_labels=y_test,
                     predictions=y_pred):
    plt.figure(figsize=(10, 7))
    
    # Detach tensors and map to numpy arrays for plotting
    plt.scatter(train_data.numpy(), train_labels.numpy(), c="b", label="Training data")
    plt.scatter(test_data.numpy(), test_labels.numpy(), c="g", label="Testing data")
    plt.scatter(test_data.numpy(), predictions.numpy(), c="r", label="Predictions")
    
    plt.legend()
    plt.show()

# Call the function to visualize results
plot_predictions()

### evaludation metrics
used to evaludate model perfromance
two main metrics:
* MAE - mean absolute error
* MSE - mean square error

In [ ]:
# ==========================================
# 4. EVALUATE THE MODEL (Equivalent to model.evaluate)
# ==========================================
my_model.eval()
with torch.inference_mode():
    # Generate test predictions
    y_pred = my_model(x_test)
    
    # PyTorch alternative to model.evaluate() tracking MAE and MSE
    mae_loss_fn = torch.nn.L1Loss()
    mse_loss_fn = torch.nn.MSELoss()
    
    test_mae = mae_loss_fn(y_pred, y_test)
    test_mse = mse_loss_fn(y_pred, y_test)
    
    print(f"Test MAE: {test_mae.item():.4f}")
    print(f"Test MSE: {test_mse.item():.4f}")

# ==========================================
# CALCULATE METRICS DIRECTLY (With squeezing)
# ==========================================
# Just like tf.squeeze(), PyTorch uses torch.squeeze() to fix dimension mismatch issues
y_pred_squeezed = torch.squeeze(y_pred)
y_test_squeezed = torch.squeeze(y_test)

# Calculate MAE manually or using the functional API
mae = torch.mean(torch.abs(y_test_squeezed - y_pred_squeezed))
print(f"Direct Squeezed MAE calculation: {mae.item():.4f}")

# Calculate MSE manually
mse = torch.mean((y_test_squeezed - y_pred_squeezed) ** 2)
print(f"Direct Squeezed MSE calculation: {mse.item():.4f}")

## Experiments and compare to improve a model

In [ ]:
# ==========================================
# 0. DEFINE METRIC FUNCTIONS & PLOTTING UNWRAPPERS
# ==========================================
def mae(y_true, y_pred):
    """Calculates Mean Absolute Error between squeezed targets."""
    return torch.mean(torch.abs(y_true.squeeze() - y_pred.squeeze())).item()

def mse(y_true, y_pred):
    """Calculates Mean Squared Error between squeezed targets."""
    return torch.mean((y_true.squeeze() - y_pred.squeeze()) ** 2).item()


In [ ]:
# Assuming x_train, y_train, x_test, y_test are already available as 2D tensors from previous blocks
# e.g., via unsqueeze(dim=1) which removes the need for tf.expand_dims

# ==========================================
# EXPERIMENT 1: model_1 (1 layer, 100 epochs)
# ==========================================
torch.manual_seed(42)


model_1 = nn.Sequential(
    nn.Linear(in_features=1, out_features=1)
)

loss_fn = nn.L1Loss()
optimizer_1 = optim.SGD(model_1.parameters(), lr=0.01)

# Fit model_1
for epoch in range(100):
    model_1.train()
    preds = model_1(x_train)
    loss = loss_fn(preds, y_train)
    optimizer_1.zero_grad()
    loss.backward()
    optimizer_1.step()

# Evaluate and predict
model_1.eval()
with torch.inference_mode():
    y_preds_1 = model_1(x_test)

plot_predictions(predictions=y_preds_1) # Using your existing plotting function

# Calculate metrics
mae_1 = mae(y_test, y_preds_1)
mse_1 = mse(y_test, y_preds_1)
print(f"Model 1 - MAE: {mae_1:.4f}, MSE: {mse_1:.4f}")


In [ ]:
# ==========================================
# EXPERIMENT 2: model_2 (2 layers, 100 epochs)
# ==========================================
torch.manual_seed(42)

model_2 = nn.Sequential(
    nn.Linear(in_features=1, out_features=1),
    nn.Linear(in_features=1, out_features=1)
)

optimizer_2 = optim.SGD(model_2.parameters(), lr=0.01)

# Fit model_2 (verbose=0 equivalent by ignoring prints)
for epoch in range(100):
    model_2.train()
    preds = model_2(x_train)
    loss = loss_fn(preds, y_train)
    optimizer_2.zero_grad()
    loss.backward()
    optimizer_2.step()

# Evaluate and predict
model_2.eval()
with torch.inference_mode():
    y_preds_2 = model_2(x_test)

plot_predictions(predictions=y_preds_2)

# Calculate metrics
mae_2 = mae(y_test, y_preds_2)
mse_2 = mse(y_test, y_preds_2)
print(f"Model 2 - MAE: {mae_2:.4f}, MSE: {mse_2:.4f}")

In [ ]:
# ==========================================
# EXPERIMENT 3: model_3 (2 layers, 500 epochs)
# ==========================================
torch.manual_seed(42)

model_3 = nn.Sequential(
    nn.Linear(in_features=1, out_features=1),
    nn.Linear(in_features=1, out_features=1)
)

optimizer_3 = optim.SGD(model_3.parameters(), lr=0.01)

# Fit model_3 for 500 epochs
for epoch in range(500):
    model_3.train()
    preds = model_3(x_train)
    loss = loss_fn(preds, y_train)
    optimizer_3.zero_grad()
    loss.backward()
    optimizer_3.step()

# Evaluate and predict
model_3.eval()
with torch.inference_mode():
    y_preds_3 = model_3(x_test)

plot_predictions(predictions=y_preds_3)

# Calculate metrics
mae_3 = mae(y_test, y_preds_3)
mse_3 = mse(y_test, y_preds_3)
print(f"Model 3 - MAE: {mae_3:.4f}, MSE: {mse_3:.4f}")

In [ ]:
# ==========================================
# COMPARE RESULTS VIA PANDAS DATAFRAME
# ==========================================
model_results = [["model_1", mae_1, mse_1],
                 ["model_2", mae_2, mse_2],
                 ["model_3", mae_3, mse_3]]

all_results = pd.DataFrame(model_results, columns=["model", "mae", "mse"])
all_results

Strange, we trained for longer but our model performed worse?

As it turns out, our model might've trained too long and has thus resulted in worse results (we'll see ways to prevent training for too long later on).

## save and load models

In [ ]:
import torch
import os

# Create a directory to store our models
os.makedirs("saved_models", exist_ok=True)

# ==========================================
# 1. Save using the standard PyTorch format (.pth / .pt)
# This saves ONLY the weights and biases (Highly Recommended)
# ==========================================
torch.save(obj=model_2.state_dict(), 
           f="saved_models/best_model_weights.pth")

# Check it out
print("Contents of saved_models directory:")
print(os.listdir("saved_models"))


# ==========================================
# 2. Save the ENTIRE model structure + weights
# (Closest equivalent to TensorFlow's HDF5/SavedModel approach)
# ==========================================
torch.save(obj=model_2, 
           f="saved_models/best_model_entire.pth")

# Check it out
print("\nVerifying entire model save:")
print(f"File exists: {os.path.exists('saved_models/best_model_entire.pth')}")


In [ ]:
# load the ENTIRE model
# ==========================================
# APPROACH 1: Load Weights Only (State Dict)
# Highly recommended. Requires recreating the model blueprint.
# ==========================================

# 1. Recreate an empty instance of the model architecture blueprint
loaded_model_weights = nn.Sequential(
    nn.Linear(in_features=1, out_features=1),
    nn.Linear(in_features=1, out_features=1)
)

# 2. Inject the saved weights into the empty blueprint
loaded_model_weights.load_state_dict(torch.load("saved_models/best_model_weights.pth", weights_only=True))
loaded_model_weights.eval() # Set to evaluation mode

print("--- Approach 1: Loaded Weights Model Summary ---")
summary(loaded_model_weights, input_size=(40, 1))

In [ ]:
# ==========================================
# APPROACH 2: Load the Entire Model Object
# Direct equivalent to tf.keras.models.load_model
# ==========================================

# Set weights_only=False because the file contains structural custom definitions
loaded_model_entire = torch.load("saved_models/best_model_entire.pth", weights_only=False)
loaded_model_entire.eval() # Set to evaluation mode

print("\n--- Approach 2: Loaded Entire Model Summary ---")
summary(loaded_model_entire, input_size=(40, 1))

In [ ]:
# ==========================================
# VERIFICATION & COMPARISON
# ==========================================
with torch.inference_mode():
    # 1. Generate predictions from the original active model instance
    model_2.eval()
    model_2_preds = model_2(x_test)
    
    # 2. Generate predictions from our newly loaded models
    weights_model_preds = loaded_model_weights(x_test)
    entire_model_preds = loaded_model_entire(x_test)

# Calculate MAEs using your custom mae() verification wrapper
mae_original = mae(y_test, model_2_preds)
mae_weights_loaded = mae(y_test, weights_model_preds)
mae_entire_loaded = mae(y_test, entire_model_preds)

# Output evaluation sanity checks
print("\n=== VERIFICATION RESULTS ===")
print(f"Original Model MAE:        {mae_original:.4f}")
print(f"Weights-Loaded Model MAE:  {mae_weights_loaded:.4f}")
print(f"Entire-Loaded Model MAE:   {mae_entire_loaded:.4f}")

print(f"\nDo original and weights-loaded model predictions match? {mae_original == mae_weights_loaded}")
print(f"Do original and entire-loaded model predictions match?  {mae_original == mae_entire_loaded}")